# Synthetic Data Creation
This builds fake energy data locally so you don't need external downloads

In [2]:
# libraries to create the dataset
import numpy as np 
import pandas as pd

In [3]:
# Generate sample dates for 1 year of hourly data
dates = pd.date_range(start = "2024-01-01", periods = 1000, freq= "h")

In [5]:
#Create realistic dummy grid load and weather variables
np.random.seed(42)
temp = np.random.normal(loc=12, scale=8, size=1000)
wind = np.abs(np.random.normal(loc=15, scale=5, size=1000))
solar = np.maximum(0, np.sin(np.linspace(0, 50, 1000)) * 100) + np.random.normal(0, 5, 1000)
solar = np.maximum(0, solar)

#Grid Load MW formula (higher load in cold weather + daily cycle)
load = 3000 + (25 - temp) * 40 - (solar * 2) + np.random.normal(0, 100, 1000)

In [6]:
#Build Pandas DataFrame and save to CSV
df_sample = pd.DataFrame({
    "timestamp": dates,
    "temperature": temp,
    "wind_speed": wind,
    "solar_radiation": solar,
    "load_mw": load
})

df_sample.to_csv("germany_energy_weather.csv", index=False)

# PySpark ML Pipeline Step-by-Step

# Step 1: Initializing PySpark, Ingestion, and Bronze Layer

In [8]:
# Initializing PySpark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('GermanEnergyForcast').master('local[*]').getOrCreate()

In [13]:
#Ingestion and Bronze Layer
raw_df = spark.read.csv('germany_energy_weather.csv', header = True, inferSchema = True)
raw_df.show()

+-------------------+-------------------+------------------+------------------+------------------+
|          timestamp|        temperature|        wind_speed|   solar_radiation|           load_mw|
+-------------------+-------------------+------------------+------------------+------------------+
|2024-01-01 00:00:00| 15.973713224089861| 21.99677718293001|               0.0|3170.2707152486664|
|2024-01-01 01:00:00| 10.893885590630523|19.623168414563843| 4.280322317280274|3469.6454306622686|
|2024-01-01 02:00:00|  17.18150830480554| 15.29815184960087| 6.031202011304119|3259.3167104430054|
|2024-01-01 03:00:00| 24.184238851264205|11.765316111472131|13.418851843235919| 3194.561507996979|
|2024-01-01 04:00:00| 10.126773002213312|18.491116568067948|10.418480298980928| 3629.747431766881|
|2024-01-01 05:00:00| 10.126904344406556|16.967426927108747| 25.83111074497831|3409.7134478237917|
|2024-01-01 06:00:00|  24.63370252405913|19.475966100138663|29.586735493466513| 3004.082056992466|
|2024-01-0

In [12]:
raw_df.write.mode('overwrite').parquet('bronze_energy_data')

# Step 2: Transformation and Silver Layer

In [16]:
from pyspark.sql.functions import col, hour, month, dayofweek, when
from pyspark.ml.feature import VectorAssembler, StandardScaler

bronze_df = spark.read.parquet('bronze_energy_data')

silver_df = bronze_df.dropna() \
    .withColumn("hour", hour("timestamp")) \
    .withColumn("month", month("timestamp")) \
    .withColumn("day_of_week", dayofweek("timestamp")) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), 1).otherwise(0))

silver_df.show()

+-------------------+-------------------+------------------+------------------+------------------+----+-----+-----------+----------+
|          timestamp|        temperature|        wind_speed|   solar_radiation|           load_mw|hour|month|day_of_week|is_weekend|
+-------------------+-------------------+------------------+------------------+------------------+----+-----+-----------+----------+
|2024-01-01 00:00:00| 15.973713224089861| 21.99677718293001|               0.0|3170.2707152486664|   0|    1|          2|         0|
|2024-01-01 01:00:00| 10.893885590630523|19.623168414563843| 4.280322317280274|3469.6454306622686|   1|    1|          2|         0|
|2024-01-01 02:00:00|  17.18150830480554| 15.29815184960087| 6.031202011304119|3259.3167104430054|   2|    1|          2|         0|
|2024-01-01 03:00:00| 24.184238851264205|11.765316111472131|13.418851843235919| 3194.561507996979|   3|    1|          2|         0|
|2024-01-01 04:00:00| 10.126773002213312|18.491116568067948|10.418480

# Step 3: Vector Assembly & Geature Scaling

In [18]:
feature_cols = ["temperature", "wind_speed", "solar_radiation", "hour", "month", "is_weekend"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled_df = assembler.transform(silver_df)

assembled_df.show()

+-------------------+-------------------+------------------+------------------+------------------+----+-----+-----------+----------+--------------------+
|          timestamp|        temperature|        wind_speed|   solar_radiation|           load_mw|hour|month|day_of_week|is_weekend|        raw_features|
+-------------------+-------------------+------------------+------------------+------------------+----+-----+-----------+----------+--------------------+
|2024-01-01 00:00:00| 15.973713224089861| 21.99677718293001|               0.0|3170.2707152486664|   0|    1|          2|         0|[15.9737132240898...|
|2024-01-01 01:00:00| 10.893885590630523|19.623168414563843| 4.280322317280274|3469.6454306622686|   1|    1|          2|         0|[10.8938855906305...|
|2024-01-01 02:00:00|  17.18150830480554| 15.29815184960087| 6.031202011304119|3259.3167104430054|   2|    1|          2|         0|[17.1815083048055...|
|2024-01-01 03:00:00| 24.184238851264205|11.765316111472131|13.4188518432359

In [19]:
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_df)
processed_df = scaler_model.transform(assembled_df)

processed_df.write.mode("overwrite").parquet("silver_energy_data")

# Step 4: Machine Learing Training & Model Evaluation

In [20]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

clean_data = spark.read.parquet("silver_energy_data")

In [21]:
# Split data into 80% training and 20% testing
train_data, test_data = clean_data.randomSplit([0.8, 0.2], seed=42)

In [23]:
# Define Gradient-Boosted Trees Model
gbt = GBTRegressor(featuresCol="features", labelCol="load_mw", maxDepth=5, maxIter=20)
gbt_model = gbt.fit(train_data)
gbt_model

GBTRegressionModel: uid=GBTRegressor_96cc5e13b6e8, numTrees=20, numFeatures=6

In [24]:
# Generate Predictions on Test Set
predictions = gbt_model.transform(test_data)
predictions

DataFrame[timestamp: timestamp, temperature: double, wind_speed: double, solar_radiation: double, load_mw: double, hour: int, month: int, day_of_week: int, is_weekend: int, raw_features: vector, features: vector, prediction: double]

In [25]:
# Calculate RMSE Metric
evaluator = RegressionEvaluator(labelCol="load_mw", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Model RMSE: {rmse:.2f} MW")

Model RMSE: 122.58 MW


# Step 5: Gold Layer Export

In [27]:
gold_df = predictions.select("timestamp", "load_mw", "prediction")
gold_df.show()
gold_df.write.mode("overwrite").parquet("gold_predictions")

+-------------------+------------------+------------------+
|          timestamp|           load_mw|        prediction|
+-------------------+------------------+------------------+
|2024-01-01 02:00:00|3259.3167104430054|3341.5546157409944|
|2024-01-01 06:00:00| 3004.082056992466| 2857.205339889913|
|2024-01-01 08:00:00| 3693.951031348824|3633.9785898390846|
|2024-01-01 13:00:00|3880.7960407824266|4112.1018671077145|
|2024-01-01 19:00:00|3876.7101905660898|3792.4825728043375|
|2024-01-01 23:00:00| 3921.750983770984| 3912.896716866315|
|2024-01-02 05:00:00|3423.1250931175496|3445.6881168684004|
|2024-01-02 11:00:00| 3731.026178792421| 3792.908674368915|
|2024-01-02 21:00:00| 3378.814428318192|3525.3395720040776|
|2024-01-02 22:00:00|  3634.47588887874|3312.2370864423324|
|2024-01-02 23:00:00|3006.2247455399684|3015.9562030189063|
|2024-01-03 01:00:00|4075.1708691268473|3801.8180076618623|
|2024-01-03 03:00:00| 3469.840902123153|3519.9328616816506|
|2024-01-03 07:00:00|  3068.68269643671|

# Visualization with Plotly

In [30]:
!pip install "pyarrow>=18.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 6.6 MB/s eta 0:00:00m eta 0:00:010:00:01


In [32]:
import warnings
warnings.filterwarnings("ignore")

In [33]:
import plotly.graph_objects as go

# Read Gold Parquet back and extract first 100 points as Pandas DataFrame
gold_spark_df = spark.read.parquet("gold_predictions")
plot_data = gold_spark_df.orderBy("timestamp").limit(100).toPandas()

fig = go.Figure()

# Plot line for actual load values
fig.add_trace(go.Scatter(
    x=plot_data['timestamp'], 
    y=plot_data['load_mw'], 
    mode='lines',
    name='Actual Load (MW)',
    line=dict(color='blue')
))

# Plot line for ML forecast predictions
fig.add_trace(go.Scatter(
    x=plot_data['timestamp'], 
    y=plot_data['prediction'], 
    mode='lines',
    name='ML Forecast (MW)',
    line=dict(color='orange', dash='dash')
))

fig.update_layout(
    title="German Power Grid Load Forecast (Actual vs PySpark Prediction)",
    xaxis_title="Timestamp",
    yaxis_title="Power Load (MW)",
    template="plotly_white"
)

fig.show()